# 3.2 · Storytelling con datos — DataRetail 360

**IDY1102 Estructuras y Visualización de Datos · Sección 003D · Unidad 3 · Actividad 3.2**
Equipo: Anyara Rosso · Robert Groom

| Campo | Detalle |
|---|---|
| **RA** | RA3 — Construye dashboards e informes para la toma de decisiones |
| **IL** | IL 3.3 (visualización interactiva con Python) · IL 3.4 (storytelling) |
| **Datos** | Capa Gold de nuestro DWH: constelación `fact_ventas` + `fact_inventario` |
| **KPIs** | Los cuatro de la Ficha de KPI (`docs/Ficha_KPI_DataRetail360.xlsx`), dos por tabla de hechos |

A diferencia del notebook de clase, aquí **no hay datos de ejemplo**: todo sale de las vistas semánticas
de Gold. Cada gráfico tiene un **título que comunica el mensaje**, y los tres juntos sostienen una
narrativa **SCI** (situación, complicación, idea) que termina en una acción.

## 0. Preparación

Requiere el pipeline ejecutado (`python -m src.pipeline --todo --reiniciar`) y `pip install plotly kaleido`.
Los cálculos viven en `src/storytelling.py`: así este notebook, la app Streamlit y el dashboard BI usan
**una sola definición de cada KPI** (principio de la guía 3.2.3).

In [1]:
import inspect, sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from src import storytelling as st

# Gráficos interactivos en Jupyter y VS Code.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 80)

engine = st.motor()
ventas = st.cargar_ventas(engine)          # vw_ventas: una fila por línea vendida
inventario = st.cargar_inventario(engine)  # agg_inventario_mes: mes x sucursal x categoría
print(f"Ventas: {len(ventas):,} líneas | Inventario: {len(inventario):,} filas mensuales "
      f"| {ventas['anio_mes'].min()} a {ventas['anio_mes'].max()}")

Ventas: 41,053 líneas | Inventario: 714 filas mensuales | 2025-01 a 2026-06


## 1. Los cuatro KPIs del semestre

Comparamos el primer semestre de 2026 con el de 2025, con las metas y umbrales de la Ficha de KPI.

| ID | KPI | Tabla de hechos | Meta |
|---|---|---|---|
| K1 | Venta neta | `fact_ventas` | ≥ mismo período del año anterior + 5% |
| K2 | Ticket promedio | `fact_ventas` | ≥ promedio de los 3 meses anteriores + 3% |
| K3 | Tasa de quiebre de stock | `fact_inventario` | ≤ 5% (alerta sobre 8%) |
| K4 | Días de cobertura | `fact_inventario` | Entre 15 y 45 días |

In [2]:
kpis = st.tabla_kpis(ventas, inventario)
kpis

,ID,KPI,Hecho,1er sem. 2026,1er sem. 2025,Meta,Estado,Variación
0,K1,Venta neta,fact_ventas,$2.112 MM,$2.021 MM,≥ +5% vs año anterior,Observación,"+4,5%"
1,K2,Ticket promedio,fact_ventas,$257.618,$253.051,Jun: ≥ $269.209 (3 meses previos + 3%),Observación,"+1,8%"
2,K3,Tasa de quiebre de stock,fact_inventario,"6,69%","6,76%",≤ 5%,Observación,"−0,08 pts"
3,K4,Días de cobertura,fact_inventario,"33,2 días","27,7 días",15 a 45 días,En meta,"+5,5 días"


**Lectura:** las ventas crecen pero bajo la meta (K1 y K2 en observación); la cobertura está sana (K4),
pero el quiebre sigue sobre la meta (K3). Los gráficos explican por qué.

## 2. Gráfico 1 · Situación (K1)

**Mensaje:** la venta crece, pero el crecimiento no es "orgánico": viene de la tienda nueva de Iquique,
que abrió en marzo de 2026. Por eso mostramos una tercera línea, **2026 sin Iquique** (naranjo, punteada),
contra la referencia de 2025 (gris).

In [3]:
fig1 = st.fig_venta_semestre(ventas)
fig1.show()

## 3. Gráfico 2 · Complicación (K3)

**Mensaje:** el quiebre de stock está sobre la meta todos los meses, no es un problema puntual.
La línea gris punteada es la meta (5%) y la roja, la alerta (8%); solo rotulamos los meses en alerta
para destacar lo importante y atenuar el resto.

In [4]:
fig2 = st.fig_quiebre_mensual(inventario)
fig2.show()

## 4. Gráfico 3 · De la complicación a la idea (K3 + K4)

**Mensaje:** el quiebre se concentra en la zona Norte (naranjo), que quiebra el doble que la zona
Centro **teniendo una cobertura parecida**. Si hubiera poco stock en total, la cobertura sería baja;
como no lo es, el problema es **reponer a tiempo** donde los plazos son más largos.
Cada barra muestra la tasa de quiebre del semestre y los días de cobertura a junio.

In [5]:
fig3 = st.fig_quiebre_sucursal(st.periodo(inventario, 2026))
fig3.show()

## 5. ¿Cuánto cuesta el quiebre? (estimación)

Para dimensionar la complicación valorizamos cada día-producto-sucursal en quiebre con la venta diaria
promedio de ese producto en esa sucursal, a precio de lista. **No es un KPI oficial**: es una estimación
para la conversación con la gerencia.

In [6]:
riesgo = st.venta_en_riesgo(engine, "2026-01", "2026-06")
crecimiento = (st.k1_venta_neta(st.periodo(ventas, 2026)) - st.k1_venta_neta(st.periodo(ventas, 2025)))
display(riesgo.assign(venta_en_riesgo=riesgo["venta_en_riesgo"].map(st.millones)))
print("Venta en riesgo del semestre:", st.millones(riesgo["venta_en_riesgo"].sum()),
      "| crecimiento de la venta neta:", st.millones(crecimiento))

,zona,venta_en_riesgo
0,Norte,$49 MM
1,Centro,$43 MM
2,Sur,$36 MM
3,Online,$24 MM


Venta en riesgo del semestre: $151 MM | crecimiento de la venta neta: $91 MM


## 6. Narrativa SCI

Los tres gráficos son la **evidencia**; la narrativa es el hilo. Se genera con las cifras de Gold,
así no queda desactualizada si los datos cambian.

In [7]:
sci = st.narrativa_sci(ventas, inventario, riesgo)
# Como HTML y con la clase tex2jax_ignore: así Jupyter no lee los signos $ como fórmulas.
parrafos = "".join(f"<p><b>{parte}.</b> {texto}</p>" for parte, texto in sci.items())
display(HTML(f'<div class="tex2jax_ignore">{parrafos}</div>'))

**La única acción que recomendamos** (pregunta desafiante de la clase): subir el stock mínimo y la
frecuencia de reposición en el Norte, partiendo por Iquique. Es medible con K3 cada mes y no exige comprar
más stock, porque la cobertura (K4) ya está dentro de la meta.

## 7. Cómo está hecho un gráfico con Plotly Express

Ejemplo: el gráfico 2. `px.line` dibuja la serie; `add_hline` agrega meta y alerta; el título se arma con
los datos para que el mensaje siempre sea verdadero.

In [8]:
print(inspect.getsource(st.fig_quiebre_mensual))

def fig_quiebre_mensual(inventario: pd.DataFrame, resaltar: tuple[str, str] | None = None) -> go.Figure:
    """Complicación (K3): tasa de quiebre mes a mes contra la meta y la alerta.

    `resaltar` = (anio_mes desde, anio_mes hasta) marca con una franja el período en análisis.
    """
    mensual = (inventario.groupby("anio_mes")[["dias_quiebre", "registros"]].sum()
               .assign(tasa=lambda d: 100 * d["dias_quiebre"] / d["registros"]).reset_index())
    mensual["mes"] = [f"{MESES[int(m[5:]) - 1]} {m[2:4]}" for m in mensual["anio_mes"]]
    sobre_meta = int((mensual["tasa"] > META_QUIEBRE).sum())
    cuantos = f"los {len(mensual)} meses" if sobre_meta == len(mensual) else f"{sobre_meta} de {len(mensual)} meses"
    titulo = (f"El quiebre de stock superó la meta de {num(META_QUIEBRE)}% en {cuantos}"
              if sobre_meta else f"El quiebre de stock se mantuvo en la meta de {num(META_QUIEBRE)}%")
    alertas = mensual[mensual["tasa"] > ALERTA_QUIEBRE]
    subtitulo = f"K

## 8. Publicar como app con Streamlit (opcional)

`app.py`, en la raíz del repositorio, muestra los cuatro KPIs, los tres gráficos, la tabla de productos
en quiebre y la narrativa, con filtros de período, zona, categoría y canal. Se ejecuta en la terminal
(no dentro del notebook):

```bash
streamlit run app.py
```

## 9. Verificación de cifras

Los KPIs del módulo deben coincidir con las vistas oficiales de Gold.

In [9]:
k1_vista = pd.read_sql("SELECT SUM(venta_neta) FROM vw_kpi_mensual WHERE anio_mes BETWEEN '2026-01' AND '2026-06'", engine).iat[0, 0]
k3_vista = pd.read_sql("SELECT ROUND(100.0 * SUM(dias_quiebre) / SUM(registros), 4) FROM vw_kpi_inventario_mensual WHERE anio_mes >= '2026-01'", engine).iat[0, 0]
k4_vista = pd.read_sql("SELECT dias_cobertura FROM vw_kpi_inventario_mensual WHERE anio_mes = '2026-06'", engine).iat[0, 0]
assert st.k1_venta_neta(st.periodo(ventas, 2026)) == k1_vista
assert round(st.k3_quiebre(st.periodo(inventario, 2026)), 4) == k3_vista
assert round(st.k4_cobertura(st.periodo(inventario, 2026)), 1) == k4_vista
print("OK: K1, K3 y K4 coinciden con vw_kpi_mensual y vw_kpi_inventario_mensual")

OK: K1, K3 y K4 coinciden con vw_kpi_mensual y vw_kpi_inventario_mensual


In [10]:
salida = RAIZ / "docs" / "storytelling"
salida.mkdir(exist_ok=True)
for nombre, fig in {"g1_venta_semestre": fig1, "g2_quiebre_mensual": fig2, "g3_quiebre_sucursal": fig3}.items():
    fig.write_html(salida / f"{nombre}.html", include_plotlyjs="cdn")
(salida / "narrativa_sci.md").write_text(
    "# Narrativa SCI · DataRetail 360 · 1er semestre 2026\n\n"
    + "\n\n".join(f"**{p}.** {t}" for p, t in sci.items()) + "\n", encoding="utf-8")
print("Gráficos (HTML interactivo) y narrativa guardados en", salida.relative_to(RAIZ))
print("Las imágenes PNG para el informe se generan en la terminal: python -m src.storytelling")

Gráficos (HTML interactivo) y narrativa guardados en docs/storytelling
Las imágenes PNG para el informe se generan en la terminal: python -m src.storytelling


---
*IDY1102 · Escuela de Informática y Telecomunicaciones · Duoc UC*